# Phase 2.1b-2.2 Colab Runner

按顺序运行三个实验组：
1. Phase 2.1b Alignment，验证 Phase 2 runner 与 Phase 1.5 基线一致。
2. Loss-scale Diagnostics，检查 top 3 loss 的分量量级。
3. Gamma refinement，精调 robust M2 的 gamma 值。

所有结果、checkpoint、日志和报告都写入 Google Drive。每组运行后立即生成可读报告，支持 `--skip-existing` 和 `--resume-mode auto` 断点续跑。

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail

# 2. Clone/update repo and install dependencies
REPO_DIR=/content/FYP
BRANCH=phase2-fixes

if [ -e "${REPO_DIR}" ] && [ ! -d "${REPO_DIR}/.git" ]; then
  mv "${REPO_DIR}" "${REPO_DIR}.backup.$(date +%Y%m%d_%H%M%S)"
fi

if [ ! -d "${REPO_DIR}/.git" ]; then
  git clone https://github.com/ROUCHER27/FYP.git "${REPO_DIR}"
fi

cd "${REPO_DIR}"
git fetch origin
git checkout "${BRANCH}"
git pull --ff-only origin "${BRANCH}"

if [ ! -f ".deps_installed_${BRANCH}" ]; then
  if [ -f requirements.txt ]; then
    python -m pip install -q -r requirements.txt
  else
    python -m pip install -q numpy pandas matplotlib seaborn torch pytest
  fi
  touch ".deps_installed_${BRANCH}"
fi

echo "Branch: $(git branch --show-current)"
echo "Commit: $(git rev-parse --short HEAD)"
test -f run_phase2_1b_alignment.py
test -f run_loss_scale_diagnostics.py
test -f run_phase2_gamma_refinement.py


In [ ]:
%%bash
set -euo pipefail

# 3. Verify Drive paths
DRIVE_ROOT=/content/drive/MyDrive/FYP

mkdir -p \
  "${DRIVE_ROOT}/phase2_1b/results" \
  "${DRIVE_ROOT}/phase2_1b/checkpoints" \
  "${DRIVE_ROOT}/phase2_1b/logs" \
  "${DRIVE_ROOT}/phase2_1b/reports" \
  "${DRIVE_ROOT}/phase2_2/loss_scale/results" \
  "${DRIVE_ROOT}/phase2_2/loss_scale/checkpoints" \
  "${DRIVE_ROOT}/phase2_2/loss_scale/logs" \
  "${DRIVE_ROOT}/phase2_2/loss_scale/reports" \
  "${DRIVE_ROOT}/phase2_2/gamma_refinement/results" \
  "${DRIVE_ROOT}/phase2_2/gamma_refinement/checkpoints" \
  "${DRIVE_ROOT}/phase2_2/gamma_refinement/logs" \
  "${DRIVE_ROOT}/phase2_2/gamma_refinement/reports"

echo "Drive root: ${DRIVE_ROOT}"
find "${DRIVE_ROOT}/phase2_1b" "${DRIVE_ROOT}/phase2_2" -maxdepth 2 -type d | sort


## Smoke Test

先用同一套 runner 路径跑一个短任务，确认代码、数据、Drive 输出和 checkpoint 路径都可用。

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP
DRIVE_ROOT=/content/drive/MyDrive/FYP

python run_phase2_1b_alignment.py \
  --losses imadl \
  --seeds 42 \
  --caps 0.05 \
  --data-dir /content/FYP \
  --test-months 2 \
  --max-epochs 2 \
  --batch-size 1024 \
  --output-root "${DRIVE_ROOT}/phase2_1b/smoke/results" \
  --checkpoint-root "${DRIVE_ROOT}/phase2_1b/smoke/checkpoints" \
  --log-root "${DRIVE_ROOT}/phase2_1b/smoke/logs" \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee "${DRIVE_ROOT}/phase2_1b/smoke/logs/smoke_$(date +%Y%m%d_%H%M%S).log"

echo "Smoke summaries:"
find "${DRIVE_ROOT}/phase2_1b/smoke/results" -name 'sanity_summary_*.json' -print


## Part 1: Phase 2.1b Alignment

验证 `imadl`, `gmadl`, `hybrid_mul` 在 Phase 2 runner 下是否接近 Phase 1.5 基线。运行后立即生成 alignment 对比 CSV。

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP
DRIVE_ROOT=/content/drive/MyDrive/FYP

python run_phase2_1b_alignment.py \
  --losses imadl,gmadl,hybrid_mul \
  --seeds 42,52,62 \
  --caps 0.05 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --output-root "${DRIVE_ROOT}/phase2_1b/results" \
  --checkpoint-root "${DRIVE_ROOT}/phase2_1b/checkpoints" \
  --log-root "${DRIVE_ROOT}/phase2_1b/logs" \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee "${DRIVE_ROOT}/phase2_1b/logs/alignment_$(date +%Y%m%d_%H%M%S).log"

python compare_phase15_phase21b.py \
  --results-root "${DRIVE_ROOT}/phase2_1b/results" \
  --losses imadl,gmadl,hybrid_mul \
  --seeds 42,52,62 \
  --caps 0.05 \
  --output-dir "${DRIVE_ROOT}/phase2_1b/reports"


## Part 2: Loss-scale Diagnostics

检查 shortlist top 3 (`imadl_m2_alpha06`, `m2_robust_gamma01`, `m2_robust_gamma10`) 的 loss 分量量级是否失衡。脚本会运行 sanity checks 并写出分析报告。

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP
DRIVE_ROOT=/content/drive/MyDrive/FYP

python run_loss_scale_diagnostics.py \
  --losses imadl_m2_alpha06,m2_robust_gamma01,m2_robust_gamma10 \
  --seed 42 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --output-root "${DRIVE_ROOT}/phase2_2/loss_scale/results" \
  --checkpoint-root "${DRIVE_ROOT}/phase2_2/loss_scale/checkpoints" \
  --log-root "${DRIVE_ROOT}/phase2_2/loss_scale/logs" \
  --analysis-dir "${DRIVE_ROOT}/phase2_2/loss_scale/reports" \
  --resume-mode auto \
  2>&1 | tee "${DRIVE_ROOT}/phase2_2/loss_scale/logs/diagnostics_$(date +%Y%m%d_%H%M%S).log"


## Part 3: Gamma Refinement

测试 `gamma=0.3, 0.5, 0.7, 1.0, 1.5`，并在完成后立即聚合大表。

In [ ]:
%%bash
set -euo pipefail

cd /content/FYP
DRIVE_ROOT=/content/drive/MyDrive/FYP
LOSSES="m2_robust_gamma03,m2_robust_gamma05,m2_robust_gamma07,m2_robust_gamma10,m2_robust_gamma15"

python run_phase2_gamma_refinement.py \
  --losses "${LOSSES}" \
  --seeds 42,52,62 \
  --caps 0.05 \
  --data-dir /content/FYP \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --output-root "${DRIVE_ROOT}/phase2_2/gamma_refinement/results" \
  --checkpoint-root "${DRIVE_ROOT}/phase2_2/gamma_refinement/checkpoints" \
  --log-root "${DRIVE_ROOT}/phase2_2/gamma_refinement/logs" \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee "${DRIVE_ROOT}/phase2_2/gamma_refinement/logs/gamma_refinement_$(date +%Y%m%d_%H%M%S).log"

python aggregate_phase2_results.py \
  --results-root "${DRIVE_ROOT}/phase2_2/gamma_refinement/results" \
  --losses "${LOSSES}" \
  --seeds 42,52,62 \
  --caps 0.05 \
  --output-dir "${DRIVE_ROOT}/phase2_2/gamma_refinement/reports"


## Final Status

统计 Drive 中的完成文件，并写入 `latest_status.json`，方便断线后恢复。

In [ ]:
%%bash
set -euo pipefail

DRIVE_ROOT=/content/drive/MyDrive/FYP
STATUS_FILE="${DRIVE_ROOT}/phase2_2/latest_status.json"

ALIGNMENT_COUNT=$(find "${DRIVE_ROOT}/phase2_1b/results" -name 'sanity_summary_*.json' | wc -l)
DIAGNOSTICS_COUNT=$(find "${DRIVE_ROOT}/phase2_2/loss_scale/results" -name 'sanity_summary_*.json' | wc -l)
GAMMA_COUNT=$(find "${DRIVE_ROOT}/phase2_2/gamma_refinement/results" -name 'sanity_summary_*.json' | wc -l)

python - <<PY
import json
from pathlib import Path
status = {
    "alignment_summaries": int("${ALIGNMENT_COUNT}"),
    "diagnostic_summaries": int("${DIAGNOSTICS_COUNT}"),
    "gamma_refinement_summaries": int("${GAMMA_COUNT}"),
    "alignment_reports": "${DRIVE_ROOT}/phase2_1b/reports",
    "diagnostics_reports": "${DRIVE_ROOT}/phase2_2/loss_scale/reports",
    "gamma_reports": "${DRIVE_ROOT}/phase2_2/gamma_refinement/reports",
}
Path("${STATUS_FILE}").write_text(json.dumps(status, indent=2) + "\n")
print(json.dumps(status, indent=2))
PY

echo "Status file: ${STATUS_FILE}"
